Question 1 : Return the page_id of every page that has zero likes, sorted ascending.

In [2]:
import pandas as pd

pages = pd.DataFrame({
    "page_id": [20001, 20045, 20701, 21785, 22222],
    "page_name": ["SQL Solutions", "Brain Exercises", "Tips for Data Analysts", "Postgres Crash Course", "Pandas Pandemonium"],
})

page_likes = pd.DataFrame({
    "user_id": [111, 201, 301, 111, 266],
    "page_id": [20001, 20001, 20045, 20045, 21785],
    "liked_date": pd.to_datetime(["2022-04-08", "2022-03-12", "2022-07-24", "2022-06-05", "2022-10-17"]),
})

Expected output: 20701 and 22222

In [3]:
liked_pages = page_likes['page_id'].drop_duplicates()

base = pd.merge(pages, liked_pages, how = 'left', on = 'page_id', indicator = True)

base_filtered = base.loc[base['_merge'] == 'left_only', 'page_id'].sort_values()

base_filtered

2    20701
4    22222
Name: page_id, dtype: int64

Question 2: Find the top 2 senders by number of messages sent in August 2022. Return sender_id and message_count, ordered by message_count descending.

In [4]:
import pandas as pd

messages = pd.DataFrame({
    "message_id": range(1, 11),
    "sender_id":   [3601, 3601, 3601, 4500, 4500, 4500, 4500, 3601, 7902, 7902],
    "receiver_id": [4500, 4500, 4500, 3601, 3601, 3601, 3601, 4500, 3601, 3601],
    "sent_date": pd.to_datetime([
        "2022-08-03", "2022-08-04", "2022-08-10", "2022-08-05", "2022-08-08",
        "2022-08-15", "2022-08-20", "2022-07-30", "2022-08-12", "2022-09-02",
    ]),
})

In [5]:
aug_logs = messages.loc[
    (messages['sent_date'] >= '2022-08-01') & (messages['sent_date'] < '2022-09-01'),  # & with parentheses, upper bound < Sep 1
    ['message_id', 'sender_id']                                                         # single list of columns
]

agg_logs = (
    aug_logs.groupby('sender_id')['message_id']
            .count()
            .reset_index(name='message_count')   # name the count column here
)

agg_logs_sorted = agg_logs.sort_values('message_count', ascending=False)  # descending

agg_logs_sorted.head(2)

,sender_id,message_count
1,4500,4
0,3601,3


Question 3: For each card, find the difference between its highest and lowest monthly issued_amount. Return card_name and difference, ordered by difference descending.

In [6]:
import pandas as pd

monthly_cards_issued = pd.DataFrame({
    "card_name": ["Chase Freedom Flex"] * 3 + ["Chase Sapphire Reserve"] * 3 + ["Chase Ink Business"] * 2,
    "issue_month": [1, 2, 3, 1, 2, 3, 1, 2],
    "issue_year": [2021] * 8,
    "issued_amount": [55000, 60000, 65000, 170000, 175000, 180000, 90000, 95000],
})

In [10]:
# monthly_cards_issued.head()

card_min_max = monthly_cards_issued.groupby('card_name').agg(
    min_spend = ('issued_amount', 'min'),
    max_spend = ('issued_amount', 'max')
).reset_index()

card_min_max['difference'] = card_min_max['max_spend'] - card_min_max['min_spend']

card_min_max[['card_name', 'difference']].sort_values(by='difference', ascending=False)

,card_name,difference
0,Chase Freedom Flex,10000
2,Chase Sapphire Reserve,10000
1,Chase Ink Business,5000


Question 4: For each category, find the top 2 highest-grossing products, where grossing means total spend, using 2022 transactions only. Return category, product and total_spend, ordered by category, then by total_spend descending.

In [11]:
import pandas as pd

product_spend = pd.DataFrame({
    "category": ["appliance", "appliance", "appliance", "appliance", "appliance",
                 "electronics", "electronics", "electronics", "electronics", "electronics", "electronics"],
    "product":  ["refrigerator", "refrigerator", "washing machine", "vacuum", "microwave",
                 "wireless headset", "wireless headset", "bluetooth speaker", "smart watch", "smart watch", "tablet"],
    "user_id":  [123, 343, 234, 456, 789, 111, 222, 333, 444, 555, 666],
    "spend":    [246.00, 299.99, 219.80, 189.00, 150.00,
                 120.00, 130.00, 99.50, 80.00, 75.00, 400.00],
    "transaction_date": pd.to_datetime([
        "2022-12-15", "2022-06-05", "2022-07-08", "2022-03-02", "2021-11-11",
        "2022-01-01", "2022-05-05", "2022-04-04", "2022-08-08", "2022-09-09", "2022-10-10"]),
})

In [12]:
ledger_2022 = product_spend.loc[
    (product_spend['transaction_date'] >= '2022-01-01') & (product_spend['transaction_date'] < '2023-01-01')
]  # < Jan 1 next year, so timestamps on Dec 31 aren't dropped

product_demand = ledger_2022.groupby(['category', 'product'])['spend'].sum().reset_index(name='total_spend')

product_demand['rnk'] = (
    product_demand.groupby('category')['total_spend']
                  .rank(method='first', ascending=False)   # ascending=False so the biggest spend gets rank 1
)

top_products = (
    product_demand.loc[product_demand['rnk'] <= 2, ['category', 'product', 'total_spend']]
                  .sort_values(['category', 'total_spend'], ascending=[True, False])
)
top_products

,category,product,total_spend
0,appliance,refrigerator,545.99
2,appliance,washing machine,219.80
5,electronics,tablet,400.00
6,electronics,wireless headset,250.00


1. The ascending=False flag. pandas rank defaults to ascending, so without it you’d get the 2 lowest spenders. 
2. The method argument. It maps to the SQL functions like this:
   
pandas method	SQL equivalent	Ties
'first'	        ROW_NUMBER	    Broken by row order, so exactly N rows per group
'min'	        RANK	        Tied rows share a rank, so you can get more than N rows
'dense'	        DENSE_RANK	    Tied rows share a rank, with no gaps after

3. The date filter. <= '2022-12-31' works on this data, but < '2023-01-01' is the safer habit.

Question 5: Year-over-year spend growth

For each product, compute the total spend per year and the year-over-year growth rate compared with the previous year. Return year, product_id, curr_year_spend, prev_year_spend and yoy_rate, with yoy_rate as a percentage rounded to 2 decimals. Order by product_id, then year. A product’s first year has no previous year, so its prev_year_spend and yoy_rate should be null.

In [13]:
import pandas as pd

user_transactions = pd.DataFrame({
    "transaction_id": range(1, 13),
    "product_id": [101, 101, 101, 101, 101, 202, 202, 202, 202, 202, 202, 303],
    "spend": [1500.0, 1000.0, 1500.0, 1400.0, 2000.0,
              800.0, 700.0, 900.0, 1200.0, 300.0, 600.0, 500.0],
    "transaction_date": pd.to_datetime([
        "2019-12-31", "2020-01-05", "2020-07-15", "2021-03-03", "2022-06-30",
        "2019-05-05", "2019-08-08", "2020-02-02", "2021-04-04", "2021-12-12", "2022-01-20",
        "2022-02-02"]),
})

In [19]:
user_transactions['year'] = user_transactions['transaction_date'].dt.year

yearly_product_sales = (
    user_transactions.groupby(['year', 'product_id'])['spend']
                     .sum()
                     .reset_index(name='curr_year_spend')
                     .sort_values(['product_id', 'year'])                    # sort before shifting
)

yearly_product_sales['prev_year_spend'] = (
    yearly_product_sales.groupby('product_id')['curr_year_spend'].shift(1)   # shift, partitioned by product only
)

yearly_product_sales['yoy_rate'] = (
    (yearly_product_sales['curr_year_spend'] - yearly_product_sales['prev_year_spend'])
    / yearly_product_sales['prev_year_spend'] * 100                          # divide by prev, as a percentage
).round(2)                                                                   # assign the rounded result

yearly_product_sales[['year', 'product_id', 'curr_year_spend', 'prev_year_spend', 'yoy_rate']]

,year,product_id,curr_year_spend,prev_year_spend,yoy_rate
0,2019,101,1500.0,NaN,NaN
2,2020,101,2500.0,1500.0,66.67
4,2021,101,1400.0,2500.0,-44.00
6,2022,101,2000.0,1400.0,42.86
1,2019,202,1500.0,NaN,NaN
3,2020,202,900.0,1500.0,-40.00
5,2021,202,1500.0,900.0,66.67
7,2022,202,600.0,1500.0,-60.00
8,2022,303,500.0,NaN,NaN


In SQL, OVER (PARTITION BY x ORDER BY y) carries its own ordering. In pandas, shift and rank don’t sort within the group for you. shift just uses the current row order, which is why the sort before the shift was necessary.

Question 6: Count the users who were active in July 2022 and also active in June 2022. A user is active in a month if they have at least one event in it. Return month (7) and monthly_active_users.

In [21]:
import pandas as pd

user_actions = pd.DataFrame({
    "user_id":    [445, 445, 742, 742, 648, 777, 777, 777, 888, 888],
    "event_id":   range(1, 11),
    "event_type": ["sign-in", "like", "sign-in", "comment", "sign-in",
                   "sign-in", "like", "comment", "like", "sign-in"],
    "event_date": pd.to_datetime([
        "2022-06-05", "2022-07-12", "2022-07-03", "2022-07-04", "2022-06-10",
        "2022-06-15", "2022-07-15", "2022-07-16", "2022-05-20", "2022-07-01"]),
})

In [23]:
july_june_2022 = user_actions.loc[
    (user_actions['event_date'] >= '2022-06-01') & (user_actions['event_date'] < '2022-08-01')
]

july_june_2022 = july_june_2022.assign(month=july_june_2022['event_date'].dt.month)   # month column to pivot on

user_action = july_june_2022.pivot_table(
    index='user_id', columns='month', values='event_id', aggfunc='count'
)

both_months = user_action.dropna()          # keep users with a value in both the 6 and 7 columns

result = pd.DataFrame({'month': [7], 'monthly_active_users': [len(both_months)]})
result

# jun_users = user_actions.loc[user_actions['event_date'].dt.month == 6, 'user_id'].unique()
# jul_users = user_actions.loc[user_actions['event_date'].dt.month == 7, 'user_id'].unique()
# len(set(jul_users) & set(jun_users))   # 2

,month,monthly_active_users
0,7,2
